# fAIr predict ; minimal examples

Uses knative cold start for containers ; POST a bbox + tile URL + ONNX URI to a fAIr predict endpoint, render the returned GeoJSON.

```
uv run --with httpx --with geopandas --with matplotlib --with jupyter jupyter lab
```

In [ ]:
import geopandas as gpd
import httpx

STAC_API = "https://stac.fair.krschap.tech/stac"
IMAGE_URI = "https://tiles.openaerialmap.org/62d85d11d8499800053796c1/0/62d85d11d8499800053796c2/{z}/{x}/{y}"
BBOX = [85.51678033745037, 27.6313353660439, 85.52323021107895, 27.637438390948745]
ZOOM = 18


def _get_stac_item(collection: str, item_id: str) -> dict:
    resp = httpx.get(f"{STAC_API}/collections/{collection}/items/{item_id}")
    resp.raise_for_status()
    return resp.json()


def call_predict(model_id: str, params: dict, collection: str = "base-models") -> gpd.GeoDataFrame:
    item = _get_stac_item(collection, model_id)
    model_uri = item["assets"]["model"]["href"]
    if collection == "base-models":
        endpoint = item["assets"]["mlm:inference-endpoint"]["href"]
    else:
        base = _get_stac_item("base-models", item["properties"]["fair:base_model_id"])
        endpoint = base["assets"]["mlm:inference-endpoint"]["href"]

    resp = httpx.post(
        endpoint,
        json={
            "model_uri": model_uri,
            "image_uri": IMAGE_URI,
            "bbox": BBOX,
            "zoom": ZOOM,
            "params": params,
        },
        timeout=300,
    )
    resp.raise_for_status()
    gdf = gpd.GeoDataFrame.from_features(resp.json()["features"])
    if not gdf.empty:
        gdf.set_crs("EPSG:4326", inplace=True)
    return gdf


def plot(gdf: gpd.GeoDataFrame, title: str):
    color = next((c for c in ("label", "class") if c in gdf.columns), None)
    ax = gdf.plot(column=color, legend=color is not None, edgecolor="black", linewidth=0.3, alpha=0.6, figsize=(8, 8))
    ax.set_title(f"{title}  (n={len(gdf)})")
    return ax

## ResNet18 classification

In [ ]:
gdf = call_predict("resnet18-classification", params={"confidence_threshold": 0.5})
print(f"features: {len(gdf)}")
gdf.head()

In [ ]:
plot(gdf, "resnet18-classification")

## UNet segmentation

In [ ]:
gdf = call_predict(
    "b799e18c-5eee-455a-ac2d-56eb2ec70419",
    collection="local-models",
    params={"confidence_threshold": 0.5, "min_class_value": 1},
)
print(f"features: {len(gdf)}")
gdf.head()

In [ ]:
plot(gdf, "unet-segmentation")

## YOLO11n detection

In [ ]:
gdf = call_predict("yolo11n-detection", params={"confidence_threshold": 0.25, "iou_threshold": 0.45})
print(f"features: {len(gdf)}")
gdf.head()

In [ ]:
plot(gdf, "yolo11n-detection")